# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt

## 4. Verify Or Prepare Data

The training scripts expect `data/tribunal/tribunal_chat_100k_balanced.csv` with `text,label` columns. If only raw `chatlogs.csv` is present, run the prep script here.

In [ ]:
from pathlib import Path
import subprocess

import pandas as pd

prepared_path = Path('data/tribunal/tribunal_chat_100k_balanced.csv')
raw_path = Path('data/tribunal/chatlogs.csv')

if not prepared_path.exists():
    if not raw_path.exists():
        raise FileNotFoundError(
            'Upload tribunal_chat_100k_balanced.csv or raw chatlogs.csv to data/tribunal/.'
        )
    # Prefer subprocess over !python so this cell stays valid Python under control flow.
    subprocess.run(['python', 'prepare_tribunal.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(df['label'].value_counts().sort_index())

## 5. Train LSTM

In [ ]:
!python train.py

## 6. Run Baseline

In [ ]:
!python baseline.py

## 7. Save Checkpoint To Drive

The repository already lives in Drive if `PROJECT_DIR` points there, but this also copies the best weights into an explicit Colab artifacts folder.

In [ ]:
from pathlib import Path
import shutil

# Keep artifacts next to the project so the path matches PROJECT_DIR from cell 4.
artifact_dir = Path(PROJECT_DIR) / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

src = Path('checkpoints/best_model.pt')
dst = artifact_dir / 'best_model.pt'
if src.exists():
    shutil.copy2(src, dst)
    print(f'Copied checkpoint to {dst}')
else:
    print('No checkpoint found. Run training first.')